# AES-128: Dashboard Interactivo

Explorá el funcionamiento de AES-128 paso a paso.
Ejecutá cada celda con **Shift+Enter** (o el botón ▶).

| Sección | Contenido |
|---------|-----------|
| 1 | El Estado: la matriz 4×4 |
| 2 | GF(2⁸): aritmética de bytes |
| 3 | SubBytes — confusión |
| 4 | ShiftRows — difusión entre columnas |
| 5 | MixColumns — difusión dentro de columnas |
| 6 | Key Expansion — 11 claves de ronda |
| 7 | Cifrado completo: estado ronda por ronda |
| 8 | ECB vs CBC: experimento visual con BMP |
| 9 | Validación NIST: 18/18 tests |

In [1]:
import sys, os, copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, os.path.abspath('.'))

from aes.core import (
    SBOX, RCON, gmul,
    bytes_to_state, state_to_bytes,
    sub_bytes, shift_rows, mix_columns, add_round_key,
    key_expansion, encrypt_block, decrypt_block
)
from aes.modes import ecb_encrypt, cbc_encrypt

# Vector canonico — FIPS-197 Apendice B
KEY   = bytes.fromhex('2b7e151628aed2a6abf7158809cf4f3c')
PLAIN = bytes.fromhex('3243f6a8885a308d313198a2e0370734')
CT    = bytes.fromhex('3925841d02dc09fbdc118597196a0b32')

def mostrar_estado(state, titulo='Estado', ax=None):
    solo = (ax is None)
    if solo:
        _, ax = plt.subplots(figsize=(4, 4))
    arr = np.array([[state[r][c] for c in range(4)] for r in range(4)], dtype=float)
    ax.imshow(arr, cmap='Blues', vmin=0, vmax=255, aspect='equal')
    for r in range(4):
        for c in range(4):
            v = state[r][c]
            ax.text(c, r, f'0x{v:02X}', ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if v > 128 else '#1a1a2e',
                    fontfamily='monospace')
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels([f'c{i}' for i in range(4)], fontsize=9)
    ax.set_yticklabels([f'f{i}' for i in range(4)], fontsize=9)
    ax.set_title(titulo, fontsize=12, fontweight='bold', pad=8)
    if solo:
        plt.tight_layout(); plt.show()

print('Setup OK.')
print(f'  KEY  : {KEY.hex()}')
print(f'  PLAIN: {PLAIN.hex()}')
print(f'  CT   : {CT.hex()}')

ModuleNotFoundError: No module named 'numpy'

---
## 1. El Estado: la matriz 4×4

AES-128 opera sobre bloques de **128 bits = 16 bytes**.
Esos 16 bytes se organizan en una **matriz 4×4** llamada "estado".

El mapeo es **por columnas**: `state[fila][col] = bloque[fila + 4 × col]`

In [ ]:
state = bytes_to_state(PLAIN)
mostrar_estado(state, 'Plaintext como estado 4x4')

print('Mapeo byte -> posicion en la matriz:')
for r in range(4):
    for c in range(4):
        idx = r + 4 * c
        print(f'  state[{r}][{c}] = 0x{state[r][c]:02X}  (byte[{idx:2d}] del bloque = 0x{PLAIN[idx]:02X}')

---
## 2. GF(2⁸): la aritmética de bytes

AES necesita multiplicar bytes entre sí y que el resultado **siempre sea otro byte** (0–255).

- **Suma** = XOR bit a bit (sin acarreo)
- **Multiplicación** = multiplicación de polinomios reducida módulo `x⁸+x⁴+x³+x+1` (0x11b)
- Implementada con el **algoritmo del campesino ruso** (Russian peasant multiplication)

In [ ]:
print(f'Multiplicacion normal:  200 x 200 = {200*200}  (no cabe en 1 byte)')
print(f'En GF(2^8):  gmul(200, 200) = {gmul(200, 200)}  (0x{gmul(200,200):02X} — siempre 0-255)')
print()
print(f'  {"x":>6}  {"x2":>6}  {"x3":>6}  <- coeficientes principales de MixColumns')
for v in [0x01, 0x02, 0x53, 0x80, 0xAB, 0xFF]:
    print(f'  0x{v:02X}   0x{gmul(v,2):02X}   0x{gmul(v,3):02X}')
print()
x = 0x53
inv_x = next(b for b in range(1, 256) if gmul(x, b) == 1)
print(f'Inverso: gmul(0x{x:02X}, 0x{inv_x:02X}) = {gmul(x, inv_x)}  (la S-box usa este inverso)')

---
## 3. SubBytes — Confusión

Reemplaza **cada byte** del estado con el valor de la **S-box** (tabla de 256 entradas precalculada).

- **Única transformación no lineal** de AES
- Construida con la inversa multiplicativa en GF(2⁸) + una transformación afín
- No se puede expresar como función matemática simple → resistencia al criptoanálisis lineal

In [ ]:
s_antes   = bytes_to_state(PLAIN)
s_despues = bytes_to_state(PLAIN)
sub_bytes(s_despues)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
mostrar_estado(s_antes,   'Antes de SubBytes',   ax1)
mostrar_estado(s_despues, 'Despues de SubBytes', ax2)
plt.suptitle('SubBytes: sustitucion byte a byte (S-box)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('Ejemplos:')
for r in range(2):
    for c in range(2):
        v = s_antes[r][c]
        print(f'  state[{r}][{c}]: 0x{v:02X}  ->  SBOX[0x{v:02X}] = 0x{SBOX[v]:02X}')

---
## 4. ShiftRows — Difusión entre columnas

Rota cada **fila** hacia la izquierda `r` posiciones:

```
Fila 0: [ a0  a1  a2  a3 ]  ->  [ a0  a1  a2  a3 ]  (sin cambio)
Fila 1: [ b0  b1  b2  b3 ]  ->  [ b1  b2  b3  b0 ]  (rota 1)
Fila 2: [ c0  c1  c2  c3 ]  ->  [ c2  c3  c0  c1 ]  (rota 2)
Fila 3: [ d0  d1  d2  d3 ]  ->  [ d3  d0  d1  d2 ]  (rota 3)
```

Resultado: cada columna contiene bytes de distintas columnas originales.

In [ ]:
s_antes = bytes_to_state(PLAIN)
sub_bytes(s_antes)
s_despues = copy.deepcopy(s_antes)
shift_rows(s_despues)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
mostrar_estado(s_antes,   'Antes de ShiftRows',   ax1)
mostrar_estado(s_despues, 'Despues de ShiftRows', ax2)
plt.suptitle('ShiftRows: rotacion ciclica de filas', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

for r in range(4):
    b = [f'0x{s_antes[r][c]:02X}'   for c in range(4)]
    d = [f'0x{s_despues[r][c]:02X}' for c in range(4)]
    print(f'Fila {r} (rot {r}): {b}  ->  {d}')

---
## 5. MixColumns — Difusión dentro de columnas

Multiplica cada **columna** del estado por una matriz fija en GF(2⁸):

```
[ 2  3  1  1 ]
[ 1  2  3  1 ]  x  columna    (todo en GF(2^8))
[ 1  1  2  3 ]
[ 3  1  1  2 ]
```

Cada byte de salida depende de los **4 bytes de esa columna**.
Junto con ShiftRows: difusión completa tras 2 rondas.

In [ ]:
s_antes = bytes_to_state(PLAIN)
sub_bytes(s_antes); shift_rows(s_antes)
s_despues = copy.deepcopy(s_antes)
mix_columns(s_despues)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
mostrar_estado(s_antes,   'Antes de MixColumns',   ax1)
mostrar_estado(s_despues, 'Despues de MixColumns', ax2)
plt.suptitle('MixColumns: multiplicacion de matriz en GF(2^8)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

s0, s1, s2, s3 = [s_antes[r][0] for r in range(4)]
out0 = gmul(2,s0) ^ gmul(3,s1) ^ s2 ^ s3
print('Verificacion manual — columna 0, fila 0:')
print(f'  gmul(2,0x{s0:02X}) ^ gmul(3,0x{s1:02X}) ^ 0x{s2:02X} ^ 0x{s3:02X} = 0x{out0:02X}')
print(f'  Resultado del codigo:                                          0x{s_despues[0][0]:02X}')
print(f'  Coincide: {out0 == s_despues[0][0]}')

---
## 6. Key Expansion — 11 claves de ronda

A partir de los 16 bytes de clave se generan **44 palabras de 32 bits** (= 11 claves de ronda).

Para `w[i]` con `i` múltiplo de 4:
```
temp = RotWord(w[i-1])  ->  SubWord(temp)  ->  temp[0] ^= Rcon[i/4-1]
w[i] = w[i-4] XOR temp
```
Para el resto: `w[i] = w[i-4] XOR w[i-1]`

In [ ]:
round_keys = key_expansion(KEY)

fig, axes = plt.subplots(1, 11, figsize=(22, 3))
for rnd in range(11):
    ax  = axes[rnd]
    rk  = round_keys[rnd]
    arr = np.array([[rk[r][c] for c in range(4)] for r in range(4)], dtype=float)
    ax.imshow(arr, cmap='YlOrRd', vmin=0, vmax=255, aspect='equal')
    for r in range(4):
        for c in range(4):
            v = rk[r][c]
            ax.text(c, r, f'{v:02X}', ha='center', va='center',
                    fontsize=7, fontweight='bold', fontfamily='monospace',
                    color='white' if v > 160 else '#1a1a2e')
    ax.set_title(f'RK{rnd}', fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Key Expansion: 11 claves de ronda (RK0 = clave original)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

w3      = [round_keys[0][r][3] for r in range(4)]
rot     = w3[1:] + w3[:1]
sub     = [SBOX[b] for b in rot]
sub[0] ^= RCON[0]
w0      = [round_keys[0][r][0] for r in range(4)]
w4      = [w0[i] ^ sub[i] for i in range(4)]
print('Primer paso: w[4] calculado vs. FIPS-197 Apendice A.1:')
print(f'  calculado: {[hex(b) for b in w4]}')
print(f'  en RK1:    {[hex(round_keys[1][r][0]) for r in range(4)]}')

---
## 7. Cifrado completo: estado ronda por ronda

```
bytes_to_state()
AddRoundKey(rk[0])
x9 rondas: SubBytes -> ShiftRows -> MixColumns -> AddRoundKey(rk[i])
Ronda 10:  SubBytes -> ShiftRows -> AddRoundKey(rk[10])   <- sin MixColumns
state_to_bytes()
```

Usamos el vector canónico de FIPS-197 Apéndice B para ver cómo evoluciona el estado.

In [ ]:
def encrypt_verbose(block, key):
    rks   = key_expansion(key)
    state = bytes_to_state(block)
    snaps = [('Entrada', copy.deepcopy(state))]
    add_round_key(state, rks[0])
    snaps.append(('ARK(0)', copy.deepcopy(state)))
    for rnd in range(1, 10):
        sub_bytes(state); shift_rows(state); mix_columns(state); add_round_key(state, rks[rnd])
        snaps.append((f'Ronda {rnd}', copy.deepcopy(state)))
    sub_bytes(state); shift_rows(state); add_round_key(state, rks[10])
    snaps.append(('Salida', copy.deepcopy(state)))
    return snaps

snaps = encrypt_verbose(PLAIN, KEY)
ncols = 6
nrows = (len(snaps) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
axes = axes.flatten()

for i, (label, st) in enumerate(snaps):
    ax  = axes[i]
    arr = np.array([[st[r][c] for c in range(4)] for r in range(4)], dtype=float)
    ax.imshow(arr, cmap='Blues', vmin=0, vmax=255, aspect='equal')
    for r in range(4):
        for c in range(4):
            v = st[r][c]
            ax.text(c, r, f'{v:02X}', ha='center', va='center', fontsize=8,
                    fontweight='bold', fontfamily='monospace',
                    color='white' if v > 128 else '#1a1a2e')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

for i in range(len(snaps), len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Estado en cada etapa — FIPS-197 Apendice B', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

result = encrypt_block(PLAIN, KEY)
print(f'Resultado: {result.hex()}')
print(f'Esperado:  {CT.hex()}')
print(f'Correcto:  {result == CT}')

---
## 8. ECB vs CBC: experimento visual

Ciframos los píxeles de una imagen BMP sintética (128×128 px, grilla 4×4 de colores) con ambos modos.

- **ECB**: bloques de plaintext idénticos → ciphertext idéntico → **estructura original visible**
- **CBC**: cada bloque se hace XOR con el ciphertext anterior → **ruido aleatorio**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, path, label, color in zip(
    axes,
    ['demo/original.png', 'demo/ecb_encrypted.png', 'demo/cbc_encrypted.png'],
    ['Original', 'ECB — INSEGURO', 'CBC — seguro'],
    ['#2e7d32',  '#c62828',        '#1565c0']
):
    ax.imshow(Image.open(path))
    ax.set_title(label, fontsize=14, fontweight='bold', color=color, pad=10)
    ax.axis('off')

plt.suptitle('Cifrado de imagen BMP: ECB vs CBC', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

# Demostracion numerica: mismo bloque -> mismo ciphertext en ECB
bloque = bytes([50, 50, 220] * 5 + [50])   # 16 bytes de rojo puro
enc_1  = ecb_encrypt(bloque, KEY)[:16]
enc_2  = ecb_encrypt(bloque, KEY)[:16]
print('ECB: mismo bloque -> mismo ciphertext')
print(f'  bloque: {bloque.hex()}')
print(f'  enc 1:  {enc_1.hex()}')
print(f'  enc 2:  {enc_2.hex()}')
print(f'  Iguales: {enc_1 == enc_2}  <- por eso ECB revela patrones')

---
## 9. Validación NIST: 18/18 tests

| Suite | Descripcion | Tests |
|-------|-------------|-------|
| FIPS-197 Apéndice B   | Bloque único + valores intermedios ronda por ronda | 2 |
| FIPS-197 Apéndice C.1 | Segundo vector de bloque único                     | 2 |
| NIST SP 800-38A ECB   | 4 bloques, cifrado y descifrado                    | 10 |
| NIST SP 800-38A CBC   | 4 bloques con IV, cifrado y descifrado             | 2 |
| Roundtrip             | `decrypt(encrypt(msg)) == msg` con padding PKCS#7  | 2 |

La coincidencia exacta con el Apéndice B — que publica el estado interno ronda por ronda — confirma que **cada transformación fue implementada correctamente**.

In [ ]:
import subprocess

r = subprocess.run(
    [sys.executable, '-m', 'tests.tests'],
    capture_output=True, text=True
)
print(r.stdout)
if r.returncode == 0:
    print('18/18 PASS — implementacion verificada contra los vectores oficiales del NIST.')
else:
    print('ERROR:', r.stderr)